In [1]:
import os
import re
import json
import joblib
import numpy as np
import pandas as pd
import faiss

from pathlib import Path
from sentence_transformers import SentenceTransformer

In [2]:
from pathlib import Path

BASE_DIR = Path.cwd()

DATA_PATH = BASE_DIR / "data" / "model_ready_jobs.csv"

MODEL_DIR = BASE_DIR / "models" / "semantic_hybrid"

MODEL_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("Data path:", DATA_PATH)
print("Model directory:", MODEL_DIR)

Data path: d:\Mota\rock-n-roll-discover-server\data\model_ready_jobs.csv
Model directory: d:\Mota\rock-n-roll-discover-server\models\semantic_hybrid


In [ ]:
df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)

display(df.head())

Dataset shape: (15841, 22)


,model_job_id,s_no,experience,job_description,job_desig,job_type,key_skills,location,salary,key_skills_normalized,...,experience_max,experience_mid,salary_min,salary_max,salary_mid,normalized_skills,experience_min.1,experience_max.1,experience_mid.1,job_profile
0,0,1,6-10 yrs,must be knowledgeable and experienced in relat...,oracle ebs technical consultant,NaN,"oracle sql, plsql, pl, oracle forms, oracle re...","bengaluru, chennai",6to10,"oracle sql, plsql, pl, oracle forms, oracle re...",...,NaN,NaN,NaN,NaN,NaN,"oracle sql, plsql, pl, oracle forms, oracle re...",6.0,10.0,8.0,job title: oracle ebs technical consultant job...
1,1,2,8-12 yrs,- experience in building scalable and highly a...,staff software engineer - object oriented anal...,NaN,"javascript, html, jquery, web development, pla...",bengaluru,10to15,"javascript, html, jquery, web development, pla...",...,NaN,NaN,NaN,NaN,NaN,"javascript, html, jquery, web development, pla...",8.0,12.0,10.0,job title: staff software engineer - object or...
2,2,3,3-8 yrs,NaN,servicenow developer,NaN,"snow, servicenow, servicenow developer","hyderabad, pune",0to3,"snow, servicenow, servicenow developer",...,NaN,NaN,NaN,NaN,NaN,"snow, servicenow, servicenow developer",3.0,8.0,5.5,job title: servicenow developer required skill...
3,3,4,3-6 yrs,our software engineers at fiserv bring an open...,"software development engineering, analyst",NaN,"qtp, flex, lan, application support, performan...",pune,6to10,"qtp, flex, lan, application support, performan...",...,NaN,NaN,NaN,NaN,NaN,"qtp, flex, lan, application support, performan...",3.0,6.0,4.5,"job title: software development engineering, a..."
4,4,5,5-10 yrs,- more than 5 years experience in hypermesh fo...,nvh engineer - nastran/ ls-dyna,NaN,"hypermesh, ls - dyna, nastran, abaqus, fatigue...","mumbai, bengaluru, hyderabad, pune, delhi",10to15,"hypermesh, ls - dyna, nastran, abaqus, fatigue...",...,NaN,NaN,NaN,NaN,NaN,"hypermesh, ls - dyna, nastran, abaqus, fatigue...",5.0,10.0,7.5,job title: nvh engineer - nastran/ ls-dyna job...


In [4]:
required_columns = [
    "model_job_id",
    "job_profile",
    "normalized_skills",
    "experience_min",
    "experience_max"
]

missing = [
    col for col in required_columns
    if col not in df.columns
]

if missing:
    raise ValueError(
        f"Missing required columns: {missing}"
    )

print("All required columns are available.")
print("Total jobs:", len(df))

All required columns are available.
Total jobs: 15841


In [5]:
MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

embedding_model = SentenceTransformer(
    MODEL_NAME
)

print("Semantic model loaded:")
print(MODEL_NAME)

Semantic model loaded:
sentence-transformers/all-MiniLM-L6-v2


In [6]:
job_texts = (
    df["job_profile"]
    .fillna("")
    .astype(str)
    .tolist()
)

print("Generating embeddings for", len(job_texts), "jobs...")

job_embeddings = embedding_model.encode(
    job_texts,
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True,
    convert_to_numpy=True
)

print("Embedding shape:", job_embeddings.shape)

Generating embeddings for 15841 jobs...


Batches:   0%|          | 0/248 [00:00<?, ?it/s]

Embedding shape: (15841, 384)


In [7]:
embedding_path = MODEL_DIR / "job_embeddings.npy"

np.save(
    embedding_path,
    job_embeddings
)

print("Saved embeddings to:")
print(embedding_path)

Saved embeddings to:
d:\Mota\rock-n-roll-discover-server\models\semantic_hybrid\job_embeddings.npy


In [8]:
embedding_dimension = job_embeddings.shape[1]

index = faiss.IndexFlatIP(
    embedding_dimension
)

index.add(
    job_embeddings.astype("float32")
)

print("FAISS index created.")
print("Number of vectors:", index.ntotal)
print("Vector dimension:", embedding_dimension)

FAISS index created.
Number of vectors: 15841
Vector dimension: 384


In [9]:
faiss_path = MODEL_DIR / "jobs.faiss"

faiss.write_index(
    index,
    str(faiss_path)
)

print("FAISS index saved:")
print(faiss_path)

FAISS index saved:
d:\Mota\rock-n-roll-discover-server\models\semantic_hybrid\jobs.faiss


In [10]:
def semantic_search(
    query,
    top_k=10
):
    
    query_embedding = embedding_model.encode(
        [query],
        normalize_embeddings=True,
        convert_to_numpy=True
    )

    scores, indices = index.search(
        query_embedding.astype("float32"),
        top_k
    )

    results = df.iloc[indices[0]].copy()

    results["semantic_score"] = scores[0]

    return results

In [11]:
query = """
Data Scientist with Python,
machine learning, SQL and
deep learning experience
"""

results = semantic_search(
    query,
    top_k=10
)

display(
    results[
        [
            "model_job_id",
            "job_profile",
            "semantic_score"
        ]
    ]
)

,model_job_id,job_profile,semantic_score
4968,4968,job title: data scientist / engineer - r/pytho...,0.723661
13830,13830,job title: data scientist / engineer - r/pytho...,0.722435
7986,7986,job title: data scientist machine learning job...,0.673722
15576,15576,job title: data scientist job description: - 2...,0.656418
6958,6958,job title: data scientist - machine learning/ ...,0.654025
1662,1662,job title: data scientist - machine learning/ ...,0.653417
927,927,job title: expert - machine learning - it job ...,0.645836
14417,14417,job title: data scientist - machine learning/p...,0.635259
11314,11314,job title: python backend developer job descri...,0.631909
1444,1444,job title: machine learning scientist - python...,0.624809


In [12]:
def normalize_skill_set(skills):
    
    if pd.isna(skills):
        return set()

    skills = str(skills).lower()

    parts = re.split(
        r"[,|;/]+",
        skills
    )

    return {
        skill.strip()
        for skill in parts
        if skill.strip()
    }

In [13]:
def calculate_skill_score(
    candidate_skills,
    job_skills
):

    candidate_skills = normalize_skill_set(
        candidate_skills
    )

    job_skills = normalize_skill_set(
        job_skills
    )

    if not candidate_skills or not job_skills:
        return 0.0

    intersection = (
        candidate_skills
        & job_skills
    )

    return (
        len(intersection)
        / len(candidate_skills)
    )

In [14]:
def calculate_experience_score(
    candidate_experience,
    job_min,
    job_max
):

    if pd.isna(job_min):
        return 0.5

    if candidate_experience is None:
        return 0.5

    candidate_experience = float(
        candidate_experience
    )

    job_min = float(job_min)

    if pd.isna(job_max):
        job_max = job_min

    job_max = float(job_max)

    # Perfectly within requirement
    if (
        job_min
        <= candidate_experience
        <= job_max
    ):
        return 1.0

    # Candidate has less experience
    if candidate_experience < job_min:

        difference = (
            job_min - candidate_experience
        )

        return max(
            0.0,
            1 - difference / max(job_min, 1)
        )

    # Candidate has more experience
    difference = (
        candidate_experience - job_max
    )

    return max(
        0.0,
        1 - difference / max(candidate_experience, 1)
    )

In [15]:
def calculate_role_score(
    candidate_query,
    job_title
):

    if not job_title:
        return 0.0

    query_words = set(
        re.findall(
            r"[a-zA-Z]+",
            candidate_query.lower()
        )
    )

    title_words = set(
        re.findall(
            r"[a-zA-Z]+",
            str(job_title).lower()
        )
    )

    stop_words = {
        "with",
        "and",
        "the",
        "for",
        "from",
        "years",
        "experience",
        "in",
        "of",
        "a",
        "an"
    }

    query_words -= stop_words
    title_words -= stop_words

    if not query_words:
        return 0.0

    overlap = (
        query_words & title_words
    )

    return len(overlap) / len(query_words)

In [16]:
def hybrid_search(
    candidate_query,
    candidate_skills,
    candidate_experience,
    top_k_retrieval=30,
    top_k_final=10
):

    # --------------------------------
    # STEP 1: Semantic Retrieval
    # --------------------------------

    candidates = semantic_search(
        candidate_query,
        top_k=top_k_retrieval
    )

    scores = []

    for _, row in candidates.iterrows():

        # ----------------------------
        # Semantic score
        # ----------------------------

        semantic_score = float(
            row["semantic_score"]
        )

        # ----------------------------
        # Skill score
        # ----------------------------

        skill_score = calculate_skill_score(
            candidate_skills,
            row["normalized_skills"]
        )

        # ----------------------------
        # Experience score
        # ----------------------------

        experience_score = (
            calculate_experience_score(
                candidate_experience,
                row["experience_min"],
                row["experience_max"]
            )
        )

        # ----------------------------
        # Role score
        # ----------------------------

        role_score = 0.0

        if "job_desig" in row.index:

            role_score = calculate_role_score(
                candidate_query,
                row["job_desig"]
            )

        # ----------------------------
        # FINAL HYBRID SCORE
        # ----------------------------

        final_score = (

            0.55 * semantic_score

            + 0.25 * skill_score

            + 0.10 * experience_score

            + 0.10 * role_score

        )

        scores.append({
            "semantic_score": semantic_score,
            "skill_score": skill_score,
            "experience_score": experience_score,
            "role_score": role_score,
            "hybrid_score": final_score
        })

    score_df = pd.DataFrame(
        scores,
        index=candidates.index
    )

    candidates = candidates.join(
        score_df
    )

    # Sort by final hybrid score
    candidates = candidates.sort_values(
        "hybrid_score",
        ascending=False
    )

    return candidates.head(
        top_k_final
    )

In [17]:
candidate_query = """
Data Scientist with 3 years of experience
in Python, machine learning, deep learning,
SQL and data analysis. Looking for a
Data Scientist or Machine Learning role.
"""

candidate_skills = [
    "python",
    "machine learning",
    "deep learning",
    "sql",
    "data analysis"
]

candidate_experience = 3

In [18]:
hybrid_results = hybrid_search(
    candidate_query=candidate_query,
    candidate_skills=candidate_skills,
    candidate_experience=candidate_experience,
    top_k_retrieval=30,
    top_k_final=10
)

display(
    hybrid_results[
        [
            "model_job_id",
            "semantic_score",
            "skill_score",
            "experience_score",
            "role_score",
            "hybrid_score"
        ]
    ]
)

ValueError: The truth value of an array with more than one element is ambiguous. Use a.any() or a.all()

In [ ]:
display(
    hybrid_results[
        [
            "model_job_id",
            "job_profile",
            "semantic_score",
            "skill_score",
            "experience_score",
            "role_score",
            "hybrid_score"
        ]
    ]
)

In [ ]:
config = {
    "embedding_model": MODEL_NAME,
    "embedding_dimension": int(embedding_dimension),
    "semantic_weight": 0.55,
    "skill_weight": 0.25,
    "experience_weight": 0.10,
    "role_weight": 0.10,
    "retrieval_top_k": 30,
    "final_top_k": 10
}

with open(
    MODEL_DIR / "config.json",
    "w"
) as f:
    
    json.dump(
        config,
        f,
        indent=4
    )


# Save metadata
metadata_columns = [
    "model_job_id",
    "normalized_skills",
    "experience_min",
    "experience_max",
    "experience_mid"
]

if "job_desig" in df.columns:
    metadata_columns.append("job_desig")

metadata = df[
    metadata_columns
].copy()

metadata.to_pickle(
    MODEL_DIR / "job_metadata.pkl"
)

print("Model artifacts saved successfully.")